# PCL Detection — SemEval 2022 Task 4 (Subtask 1)

**Task**: Binary classification — does a paragraph contain Patronizing/Condescending Language (PCL)?

**Model**: `microsoft/deberta-v3-base` — stronger encoder than the RoBERTa-base baseline

**Approach**:
1. **Keyword prepending**: prefix each text with `"Topic: {keyword}."` to give the model explicit topic context
2. **WeightedRandomSampler**: oversample PCL to 25% of each batch (vs 9.5% natural rate)
3. **sqrt pos_weight ≈ 3.1**: avoids double-counting imbalance already handled by the sampler; produces natural thresholds close to 0.5
4. **Multi-task learning**: auxiliary 7-category PCL loss forces richer representations
5. **Threshold tuning** on dev set + **two-stage training** (train+dev) for test generalisation

**Baseline**: RoBERTa-base, standard loss, t=0.5 → F1 = 0.48 (dev), 0.49 (test)

In [1]:
# Install required packages — run this cell once, then restart the kernel
# !pip install transformers>=4.35.0 sentencepiece protobuf torch scikit-learn pandas numpy tqdm

# Packages needed and why:
#   transformers  — AutoTokenizer, AutoModelForSequenceClassification
#   sentencepiece — DeBERTa-v3 uses a SentencePiece tokenizer (.spm.model)
#   protobuf      — required by sentencepiece to parse the binary .spm.model file
#
# NOTE: tiktoken is NOT needed — it is for OpenAI/GPT models only

In [2]:
# ── Data setup (Colab) ────────────────────────────────────────────────────────
# Run this cell once at the start of each Colab session.

from google.colab import drive
import os, shutil

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Create & switch to a persistent working directory on Drive
WORKDIR = '/content/drive/MyDrive/pcl_chatbot'
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
print(f"Working directory: {os.getcwd()}")

# 3. Copy all data files from MyDrive/NLP_CW/
NLP_CW = '/content/drive/MyDrive/NLP_CW'
ALL_FILES = [
    'dontpatronizeme_pcl.tsv',
    'train_semeval_parids-labels.csv',
    'dev_semeval_parids-labels.csv',
    'task4_test.tsv',
]
for fname in ALL_FILES:
    if os.path.exists(fname):
        print(f"  {fname} already present — skipping")
    else:
        shutil.copy(os.path.join(NLP_CW, fname), '.')
        print(f"  Copied {fname}")

# 4. Verify
print("\nData files:")
for f in ALL_FILES:
    size = os.path.getsize(f) / 1e3
    print(f"  ✓ {f}  ({size:.0f} KB)")

Mounted at /content/drive
Working directory: /content/drive/MyDrive/pcl_chatbot
  dontpatronizeme_pcl.tsv already present — skipping
  train_semeval_parids-labels.csv already present — skipping
  dev_semeval_parids-labels.csv already present — skipping
  task4_test.tsv already present — skipping

Data files:
  ✓ dontpatronizeme_pcl.tsv  (3123 KB)
  ✓ train_semeval_parids-labels.csv  (242 KB)
  ✓ dev_semeval_parids-labels.csv  (61 KB)
  ✓ task4_test.tsv  (1145 KB)


In [3]:
import os, ast, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    f1_score, precision_score, recall_score, classification_report
)
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f"BF16   : {'supported' if bf16_ok else 'NOT supported — will use FP32'}")

Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB
BF16   : supported


In [4]:
# ── Configuration ────────────────────────────────────────────────────────────
import os

CFG = {
    # Model
    'model_name':    'microsoft/deberta-v3-base',
    # Data files
    'main_data':     'dontpatronizeme_pcl.tsv',
    'train_ids':     'train_semeval_parids-labels.csv',
    'dev_ids':       'dev_semeval_parids-labels.csv',
    'test_data':     'task4_test.tsv',
    # Tokenizer — 256 tokens captures full paragraph context
    'max_length':    256,
    # Training
    'batch_size':    12,
    'grad_accum':    3,          # effective batch = 36
    'lr':            2e-5,       # reverted — 1e-5 converged too slowly for patience=4
    'weight_decay':  0.01,
    'warmup_ratio':  0.10,
    'num_epochs':    12,
    'patience':      4,          # increased from 3 — survives the volatile middle epochs
    'aux_weight':    0.3,
    'seed':          42,
    # Output files (BestModel/ folder — required by spec Exercise 4 & 5.1)
    'best_model':    'BestModel/best_model.pt',
    'final_model':   'BestModel/final_model.pt',
    'dev_preds':     'BestModel/dev.txt',
    'test_preds':    'BestModel/test.txt',
}

PCL_CATEGORIES = [
    'Unbalanced power relations',
    'Shallow solution',
    'Presupposition',
    'Authority voice',
    'Metaphor',
    'Compassion',
    'The poorer the merrier',
]

os.makedirs('BestModel', exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(CFG['seed'])
print("Config ready.  BestModel/ folder created.")
print(f"  max_length={CFG['max_length']}  lr={CFG['lr']}  epochs={CFG['num_epochs']}  patience={CFG['patience']}")
print(f"  batch_size={CFG['batch_size']}  grad_accum={CFG['grad_accum']}  effective_batch={CFG['batch_size']*CFG['grad_accum']}")

Config ready.  BestModel/ folder created.
  max_length=256  lr=2e-05  epochs=12  patience=4
  batch_size=12  grad_accum=3  effective_batch=36


## 1. Data Loading

In [5]:
# ── Load main dataset (dontpatronizeme_pcl.tsv) ──────────────────────────────
# Uses a line-by-line parser that safely skips the 4-row disclaimer header
# and any malformed rows.

def load_main_data(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip('\n').split('\t', 5)
            if len(parts) == 6 and parts[0].strip().isdigit():
                rows.append({
                    'par_id':    int(parts[0]),
                    'keyword':   parts[2],
                    'country':   parts[3],
                    'text':      parts[4].strip(),
                    'label_raw': int(parts[5]),
                })
    df = pd.DataFrame(rows)
    # Task 3 binary label: PCL if >= 2 annotators flagged it
    df['label_binary'] = (df['label_raw'] >= 2).astype(int)
    return df


def load_split_ids(path):
    """Load par_ids and their 7-element Task-4 multi-label vectors."""
    df = pd.read_csv(path)
    df['par_id'] = df['par_id'].astype(int)
    df['label_list'] = df['label'].apply(ast.literal_eval)
    return df[['par_id', 'label_list']]


def load_test_data(path):
    """Load test set — par_ids start with 't_', no label column, no disclaimer."""
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip('\n').split('\t')
            if len(parts) >= 5 and parts[0].strip().startswith('t_'):
                rows.append({
                    'par_id':  parts[0].strip(),
                    'keyword': parts[2] if len(parts) > 2 else '',
                    'country': parts[3] if len(parts) > 3 else '',
                    'text':    parts[4].strip() if len(parts) > 4 else '',
                })
    df = pd.DataFrame(rows)
    df = df[df['text'].str.len() > 0].reset_index(drop=True)
    return df


# Load everything
df_main      = load_main_data(CFG['main_data'])
df_train_ids = load_split_ids(CFG['train_ids'])
df_dev_ids   = load_split_ids(CFG['dev_ids'])
df_test      = load_test_data(CFG['test_data'])

print(f"Main dataset : {len(df_main):>6,} rows")
print(f"Train IDs    : {len(df_train_ids):>6,} rows")
print(f"Dev IDs      : {len(df_dev_ids):>6,} rows")
print(f"Test set     : {len(df_test):>6,} rows")

Main dataset : 10,469 rows
Train IDs    :  8,375 rows
Dev IDs      :  2,094 rows
Test set     :  3,832 rows


In [6]:
# ── Merge labels into train / dev splits ─────────────────────────────────────

df_train = df_main[df_main['par_id'].isin(df_train_ids['par_id'])].copy()
df_dev   = df_main[df_main['par_id'].isin(df_dev_ids['par_id'])].copy()

df_train = df_train.merge(df_train_ids, on='par_id', how='left').reset_index(drop=True)
df_dev   = df_dev.merge(df_dev_ids,   on='par_id', how='left').reset_index(drop=True)

# Fill Task-4 labels with all-zeros for samples not in the label CSV
_zero7 = lambda x: x if isinstance(x, list) else [0] * 7
df_train['label_list'] = df_train['label_list'].apply(_zero7)
df_dev['label_list']   = df_dev['label_list'].apply(_zero7)

print(f"Train : {len(df_train):,} samples")
print(f"Dev   : {len(df_dev):,} samples")

# Task 3 class breakdown
n_neg_tr = (df_train['label_binary'] == 0).sum()
n_pos_tr = (df_train['label_binary'] == 1).sum()
n_neg_dv = (df_dev['label_binary'] == 0).sum()
n_pos_dv = (df_dev['label_binary'] == 1).sum()
print(f"\nTask 3 — Train: No-PCL={n_neg_tr:,}  PCL={n_pos_tr:,}  ratio={n_neg_tr/n_pos_tr:.1f}:1")
print(f"Task 3 — Dev  : No-PCL={n_neg_dv:,}  PCL={n_pos_dv:,}")

# ── Keyword prepending ────────────────────────────────────────────────────────
# The dataset includes a keyword column (e.g. "homeless", "migrant",
# "vulnerable") identifying which group the paragraph discusses.
# Prepending "Topic: {keyword}." gives the model an explicit signal about
# the vulnerable group — a strong prior for PCL detection — at zero cost.
def prepend_kw(row):
    kw = str(row['keyword']).strip()
    return f"Topic: {kw}. {row['text']}" if kw else row['text']

df_train['text'] = df_train.apply(prepend_kw, axis=1)
df_dev['text']   = df_dev.apply(prepend_kw, axis=1)
df_test['text']  = df_test.apply(prepend_kw, axis=1)

print(f"\nKeyword prepended to all splits.")
print(f"  Example train: {df_train['text'].iloc[0][:90]}...")

Train : 8,375 samples
Dev   : 2,094 samples

Task 3 — Train: No-PCL=7,581  PCL=794  ratio=9.5:1
Task 3 — Dev  : No-PCL=1,895  PCL=199

Keyword prepended to all splits.
  Example train: Topic: hopeless. We 're living in times of absolute insanity , as I 'm pretty sure most pe...


In [7]:
# ── Data verification ─────────────────────────────────────────────────────────

# 1. No overlap between train and dev
overlap = set(df_train['par_id']) & set(df_dev['par_id'])
print(f"Train/dev par_id overlap (should be 0): {len(overlap)}")

# 2. Task-4 label array stats
train_arr = np.array(df_train['label_list'].tolist(), dtype=np.float32)
dev_arr   = np.array(df_dev['label_list'].tolist(),   dtype=np.float32)

print(f"\nTask 4 per-category positive counts (train) — {len(train_arr):,} samples total:")
for i, cat in enumerate(PCL_CATEGORIES):
    cnt = int(train_arr[:, i].sum())
    pct = train_arr[:, i].mean() * 100
    print(f"  [{i}] {cat:<35s} : {cnt:>4} ({pct:.1f}%)")

# 3. Quick text sample
print("\nSample row:")
print(df_train[df_train['label_binary'] == 1].iloc[0][['text', 'label_binary', 'label_list']].to_string())

Train/dev par_id overlap (should be 0): 0

Task 4 per-category positive counts (train) — 8,375 samples total:
  [0] Unbalanced power relations          :  574 (6.9%)
  [1] Shallow solution                    :  160 (1.9%)
  [2] Presupposition                      :  162 (1.9%)
  [3] Authority voice                     :  192 (2.3%)
  [4] Metaphor                            :  145 (1.7%)
  [5] Compassion                          :  363 (4.3%)
  [6] The poorer the merrier              :   29 (0.3%)

Sample row:
text            Topic: disabled. Arshad said that besides lear...
label_binary                                                    1
label_list                                  [1, 0, 0, 0, 0, 0, 0]


## 2. Dataset Class & Training Utilities

In [8]:
# ── Tokenizer ────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])
print(f"Tokenizer loaded: {CFG['model_name']}")


# ── Dataset class ─────────────────────────────────────────────────────────────
class PCLDataset(Dataset):
    """
    Binary PCL dataset.  Optionally carries 7-category aux_labels for
    multi-task training (train/dev only — test set has no category labels).
    """
    def __init__(self, texts, labels, tokenizer, max_length, aux_labels=None):
        self.texts      = [str(t) for t in texts]
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.aux_labels = aux_labels   # list of 7-element lists, or None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(float(self.labels[idx]), dtype=torch.float)
        if self.aux_labels is not None:
            item['aux_labels'] = torch.tensor(self.aux_labels[idx], dtype=torch.float)
        return item

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Tokenizer loaded: microsoft/deberta-v3-base


In [9]:
# ── Training / evaluation / prediction helpers ───────────────────────────────
# Pure FP32: DeBERTa-v3 disentangled attention + StableDropout overflow in
# BF16/FP16 on Blackwell (RTX 50-series). FP32 is stable and fast enough here.
#
# Multi-task setup:
#   model has num_labels=8  →  logits shape (batch, 8)
#   logits[:, 0]   = binary PCL logit   (primary task)
#   logits[:, 1:]  = 7-category logits  (auxiliary task, weight = aux_weight)

def run_epoch(model, loader, optimizer, scheduler, criterion,
              grad_accum, aux_criterion=None, aux_weight=0.3):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(enumerate(loader), total=len(loader), desc='Train', leave=True)
    for step, batch in pbar:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        extra = {}
        if 'token_type_ids' in batch:
            extra['token_type_ids'] = batch['token_type_ids'].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask, **extra).logits

        # Primary binary loss
        loss = criterion(logits[:, 0], labels)

        # Auxiliary 7-category loss (train/dev have labels; test does not)
        if aux_criterion is not None and 'aux_labels' in batch:
            aux_labels = batch['aux_labels'].to(device)
            loss = loss + aux_weight * aux_criterion(logits[:, 1:], aux_labels)

        loss = loss / grad_accum
        loss.backward()

        if (step + 1) % grad_accum == 0 or (step + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * grad_accum
        pbar.set_postfix({'loss': f'{total_loss / (step + 1):.4f}'})

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, threshold=0.5):
    model.eval()
    all_logits, all_labels = [], []

    for batch in tqdm(loader, desc='Eval', leave=False):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        extra = {}
        if 'token_type_ids' in batch:
            extra['token_type_ids'] = batch['token_type_ids'].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask, **extra).logits
        all_logits.append(logits[:, 0].cpu().float().numpy())   # binary logit only
        all_labels.append(batch['labels'].numpy())

    all_logits = np.concatenate(all_logits, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    probs = 1.0 / (1.0 + np.exp(-all_logits))
    preds = (probs >= threshold).astype(int)
    f1    = f1_score(all_labels.astype(int), preds, zero_division=0)

    return f1, probs, all_labels


def tune_threshold(probs, labels):
    best_t, best_f = 0.5, 0.0
    for t in np.linspace(0.05, 0.95, 91):
        p = (probs >= t).astype(int)
        f = f1_score(labels.astype(int), p, zero_division=0)
        if f > best_f:
            best_f, best_t = f, t
    print(f"Optimal threshold: {best_t:.3f}  →  F1 = {best_f:.4f}")
    return float(best_t)


@torch.no_grad()
def predict(model, loader, threshold):
    model.eval()
    all_probs = []

    for batch in tqdm(loader, desc='Predict', leave=False):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        extra = {}
        if 'token_type_ids' in batch:
            extra['token_type_ids'] = batch['token_type_ids'].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask, **extra).logits
        all_probs.append(torch.sigmoid(logits[:, 0]).cpu().float().numpy())

    all_probs = np.concatenate(all_probs, axis=0)
    preds     = (all_probs >= threshold).astype(int)
    return preds, all_probs


print("Utilities defined (pure FP32 · multi-task).")

Utilities defined (pure FP32 · multi-task).


---
## 3. Binary PCL Detection

**Baseline**: RoBERTa-base, standard CE loss, t=0.5 → **F1 = 0.48 (dev), 0.49 (test)**

**Our approach**:
- **DeBERTa-v3-base**: disentangled attention + ELECTRA-style pretraining — stronger than RoBERTa-base at the same parameter count
- **Keyword prepending**: each text prefixed with `"Topic: {keyword}."` — gives the model explicit context about which vulnerable group is discussed, a strong PCL prior
- **WeightedRandomSampler** (25% positive target): oversamples PCL examples so each batch contains ~3 positives instead of ~1, providing 2.5× more gradient signal per update
- **sqrt pos_weight ≈ 3.1**: avoids double-counting the imbalance already handled by the sampler; produces thresholds close to 0.5 that generalise well to test
- **Multi-task auxiliary loss** (weight=0.3): model outputs 8 logits — logit[0] is binary PCL, logits[1–7] are the 7 PCL sub-categories; auxiliary signal forces richer representations
- **Threshold tuning** on dev set + **two-stage training**: stage 1 finds best epoch and threshold; stage 2 retrains on train+dev for the test predictions
- FP32, gradient accumulation (effective batch = 36), AdamW + linear warmup, early stopping (patience=4)

In [10]:
from torch.utils.data import WeightedRandomSampler

# ── Datasets ──────────────────────────────────────────────────────────────────
train_ds = PCLDataset(df_train['text'].tolist(),
                      df_train['label_binary'].tolist(),
                      tokenizer, CFG['max_length'],
                      aux_labels=df_train['label_list'].tolist())

dev_ds   = PCLDataset(df_dev['text'].tolist(),
                      df_dev['label_binary'].tolist(),
                      tokenizer, CFG['max_length'],
                      aux_labels=df_dev['label_list'].tolist())

test_ds  = PCLDataset(df_test['text'].tolist(),
                      [0] * len(df_test),
                      tokenizer, CFG['max_length'])

# ── WeightedRandomSampler ─────────────────────────────────────────────────────
# With a 9.5:1 ratio, a random batch of 12 contains only ~1 PCL example on
# average.  The sampler oversamples PCL so that ~25% of each batch is positive
# (3 out of 12), giving the model much richer signal per update.
n_neg = int((df_train['label_binary'] == 0).sum())
n_pos = int((df_train['label_binary'] == 1).sum())

TARGET_POS_FRAC = 0.25
w_pos = TARGET_POS_FRAC / n_pos
w_neg = (1.0 - TARGET_POS_FRAC) / n_neg
sample_weights = np.where(df_train['label_binary'].values == 1, w_pos, w_neg)
sampler = WeightedRandomSampler(
    weights=torch.FloatTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True,
)
print(f"Sampler: target {TARGET_POS_FRAC*100:.0f}% positive per batch "
      f"(was {n_pos/(n_pos+n_neg)*100:.1f}%)")

# ── Data loaders ──────────────────────────────────────────────────────────────
train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'],
                      sampler=sampler, num_workers=0, pin_memory=True)
dev_dl   = DataLoader(dev_ds,   batch_size=CFG['batch_size'] * 2,
                      shuffle=False, num_workers=0, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=CFG['batch_size'] * 2,
                      shuffle=False, num_workers=0)

# ── Loss functions ────────────────────────────────────────────────────────────
# Primary: sqrt(neg/pos) ≈ 3.1 instead of the full ratio 9.55.
# The sampler already oversamples positives 2.5× — using the full ratio on
# top would double-count the imbalance and push the threshold to extreme
# values (as seen previously with t=0.840).  sqrt gives a natural threshold
# close to 0.5, which generalises much better to the test set.
pos_weight = torch.tensor([np.sqrt(n_neg / n_pos)], dtype=torch.float).to(device)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print(f"Binary pos_weight = {pos_weight.item():.2f}  (sqrt of {n_neg/n_pos:.2f})")

# Auxiliary: per-label weights for 7 PCL categories (unchanged)
train_arr  = np.array(df_train['label_list'].tolist(), dtype=np.float32)
pos_counts = train_arr.sum(axis=0).clip(min=1)
neg_counts = len(train_arr) - pos_counts
pw_aux     = torch.tensor(neg_counts / pos_counts, dtype=torch.float).to(device)
aux_criterion = nn.BCEWithLogitsLoss(pos_weight=pw_aux)
print("Auxiliary criterion ready (7 categories).")

# ── Model ─────────────────────────────────────────────────────────────────────
set_seed(CFG['seed'])
model = AutoModelForSequenceClassification.from_pretrained(
    CFG['model_name'], num_labels=8, torch_dtype=torch.float32
).to(device)

total_steps  = (len(train_dl) // CFG['grad_accum']) * CFG['num_epochs']
warmup_steps = int(total_steps * CFG['warmup_ratio'])

optimizer = AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
print(f"\nModel: {CFG['model_name']}  |  num_labels=8")
print(f"Total steps: {total_steps} | Warmup: {warmup_steps}")

Sampler: target 25% positive per batch (was 9.5%)
Binary pos_weight = 3.09  (sqrt of 9.55)
Auxiliary criterion ready (7 categories).


`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight      


Model: microsoft/deberta-v3-base  |  num_labels=8
Total steps: 2784 | Warmup: 278


In [11]:
# ── Stage-1 Training loop (train set only) ────────────────────────────────────
# We track best_epoch so Stage 2 can replicate the same number of passes
# on the larger train+dev dataset.  Early stopping (patience=CFG['patience'])
# fires when dev F1 has not improved for that many consecutive epochs.

best_f1          = 0.0
best_epoch       = 0
best_probs       = None
best_labels      = None
patience_counter = 0

for epoch in range(CFG['num_epochs']):
    print(f"\n── Epoch {epoch + 1} / {CFG['num_epochs']} ──")
    train_loss = run_epoch(
        model, train_dl, optimizer, scheduler, criterion,
        CFG['grad_accum'],
        aux_criterion=aux_criterion,
        aux_weight=CFG['aux_weight'],
    )
    f1, probs, labels = evaluate(model, dev_dl, threshold=0.5)
    print(f"   Train loss: {train_loss:.4f}  |  Dev F1 (t=0.5): {f1:.4f}", end='')

    if f1 > best_f1:
        best_f1          = f1
        best_epoch       = epoch + 1
        best_probs       = probs.copy()
        best_labels      = labels.copy()
        patience_counter = 0
        torch.save(model.state_dict(), CFG['best_model'])
        print(f"  ✓ New best — saved (epoch {best_epoch})")
    else:
        patience_counter += 1
        print(f"  [patience {patience_counter}/{CFG['patience']}]")
        if patience_counter >= CFG['patience']:
            print(f"\nEarly stopping: no improvement for {CFG['patience']} epochs.")
            break

print(f"\nStage-1 complete.  Best dev F1 = {best_f1:.4f}  at epoch {best_epoch}")


── Epoch 1 / 12 ──


Train:   0%|          | 0/698 [00:00<?, ?it/s]

Eval:   0%|          | 0/88 [00:00<?, ?it/s]

   Train loss: 1.5100  |  Dev F1 (t=0.5): 0.3349  ✓ New best — saved (epoch 1)

── Epoch 2 / 12 ──


Train:   0%|          | 0/698 [00:00<?, ?it/s]

...

Eval:   0%|          | 0/88 [00:00<?, ?it/s]

   Train loss: 0.2136  |  Dev F1 (t=0.5): 0.5125  [patience 2/4]

── Epoch 12 / 12 ──


Train:   0%|          | 0/698 [00:00<?, ?it/s]

Eval:   0%|          | 0/88 [00:00<?, ?it/s]

   Train loss: 0.1912  |  Dev F1 (t=0.5): 0.5099  [patience 3/4]

Stage-1 complete.  Best dev F1 = 0.5225  at epoch 9


In [12]:
# ── Threshold tuning + final dev evaluation ───────────────────────────────────
model.load_state_dict(torch.load(CFG['best_model'], map_location=device))
_, best_probs, best_labels = evaluate(model, dev_dl, threshold=0.5)

print("Searching optimal threshold on dev set ...")
best_thresh = tune_threshold(best_probs, best_labels)

preds_dev = (best_probs >= best_thresh).astype(int)

print(f"\n{'=' * 55}")
print(f"  FINAL RESULTS  (threshold = {best_thresh:.3f})")
print(f"{'=' * 55}")
print(f"  Baseline (RoBERTa-base, t=0.5)   F1 = 0.4800")
final_f1 = f1_score(best_labels.astype(int), preds_dev)
print(f"  Our model (DeBERTa-v3-base)       F1 = {final_f1:.4f}")
print(f"  Improvement: +{final_f1 - 0.48:.4f}")
print()
print(classification_report(
    best_labels.astype(int), preds_dev,
    target_names=['No-PCL', 'PCL'], digits=4
))

# Save dev.txt — one prediction per line (required format, Exercise 5.1)
with open(CFG['dev_preds'], 'w') as f:
    for p in preds_dev:
        f.write(f"{int(p)}\n")
print(f"Saved → {CFG['dev_preds']}  ({len(preds_dev)} lines)")

Eval:   0%|          | 0/88 [00:00<?, ?it/s]

Searching optimal threshold on dev set ...
Optimal threshold: 0.940  →  F1 = 0.5397

  FINAL RESULTS  (threshold = 0.940)
  Baseline (RoBERTa-base, t=0.5)   F1 = 0.4800
  Our model (DeBERTa-v3-base)       F1 = 0.5397
  Improvement: +0.0597

              precision    recall  f1-score   support

      No-PCL     0.9568    0.9351    0.9458      1895
         PCL     0.4917    0.5980    0.5397       199

    accuracy                         0.9031      2094
   macro avg     0.7243    0.7665    0.7428      2094
weighted avg     0.9126    0.9031    0.9072      2094

Saved → BestModel/dev.txt  (2094 lines)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Stage-2: retrain on train + dev, generate final test predictions ──────────
# Override Stage-1 values — edit these if rerunning after a session reset
best_epoch  = 9    # best epoch found in Stage 1
best_thresh = 0.5  # use t=0.5 (more robust than dev-tuned 0.940)

print("=" * 60)
print(f"Stage 2: retraining on train + dev ({len(df_train)+len(df_dev):,} samples)")
print(f"         for {best_epoch} epoch(s)  |  threshold = {best_thresh:.3f}")
print("=" * 60)

# ── Combined dataset ──────────────────────────────────────────────────────────
df_full = pd.concat([df_train, df_dev], ignore_index=True)

full_ds = PCLDataset(
    df_full['text'].tolist(),
    df_full['label_binary'].tolist(),
    tokenizer, CFG['max_length'],
    aux_labels=df_full['label_list'].tolist(),
)

# ── WeightedRandomSampler for stage 2 ────────────────────────────────────────
n_neg_full = int((df_full['label_binary'] == 0).sum())
n_pos_full = int((df_full['label_binary'] == 1).sum())

w_pos_f = TARGET_POS_FRAC / n_pos_full
w_neg_f = (1.0 - TARGET_POS_FRAC) / n_neg_full
sw_full = np.where(df_full['label_binary'].values == 1, w_pos_f, w_neg_f)
sampler_full = WeightedRandomSampler(
    weights=torch.FloatTensor(sw_full),
    num_samples=len(sw_full),
    replacement=True,
)

full_dl = DataLoader(full_ds, batch_size=CFG['batch_size'],
                     sampler=sampler_full, num_workers=0, pin_memory=True)

# ── Loss functions for combined data ─────────────────────────────────────────
pos_weight_full = torch.tensor([np.sqrt(n_neg_full / n_pos_full)], dtype=torch.float).to(device)
criterion_full  = nn.BCEWithLogitsLoss(pos_weight=pos_weight_full)
print(f"Full-data pos_weight = {pos_weight_full.item():.2f}  (No-PCL={n_neg_full}, PCL={n_pos_full})")

full_arr      = np.array(df_full['label_list'].tolist(), dtype=np.float32)
pos_cnt_f     = full_arr.sum(axis=0).clip(min=1)
neg_cnt_f     = len(full_arr) - pos_cnt_f
pw_aux_full   = torch.tensor(neg_cnt_f / pos_cnt_f, dtype=torch.float).to(device)
aux_crit_full = nn.BCEWithLogitsLoss(pos_weight=pw_aux_full)

# ── Fresh model + optimiser ───────────────────────────────────────────────────
set_seed(CFG['seed'])
model_final = AutoModelForSequenceClassification.from_pretrained(
    CFG['model_name'], num_labels=8, torch_dtype=torch.float32
).to(device)

total_steps_s2  = (len(full_dl) // CFG['grad_accum']) * best_epoch
warmup_steps_s2 = int(total_steps_s2 * CFG['warmup_ratio'])

optimizer_s2 = AdamW(model_final.parameters(),
                     lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler_s2 = get_linear_schedule_with_warmup(
    optimizer_s2,
    num_warmup_steps=warmup_steps_s2,
    num_training_steps=total_steps_s2,
)
print(f"Stage-2 steps: {total_steps_s2}  |  warmup: {warmup_steps_s2}\n")

# ── Train ─────────────────────────────────────────────────────────────────────
for epoch in range(best_epoch):
    print(f"── Stage-2 Epoch {epoch + 1} / {best_epoch} ──")
    run_epoch(
        model_final, full_dl, optimizer_s2, scheduler_s2, criterion_full,
        CFG['grad_accum'],
        aux_criterion=aux_crit_full,
        aux_weight=CFG['aux_weight'],
    )

torch.save(model_final.state_dict(), CFG['final_model'])
print(f"\nStage-2 model saved → {CFG['final_model']}")

# ── Test predictions ──────────────────────────────────────────────────────────
preds_test, probs_test = predict(model_final, test_dl, threshold=best_thresh)

with open(CFG['test_preds'], 'w') as f:
    for p in preds_test:
        f.write(f"{int(p)}\n")

print(f"Saved → {CFG['test_preds']}  ({len(preds_test)} lines)")
print(f"Predicted PCL: {preds_test.sum()} / {len(preds_test)}  ({preds_test.mean()*100:.1f}%)")
print(f"\nThreshold used: {best_thresh:.3f}")
print("\nBestModel/ contents:")
for fn in sorted(os.listdir('BestModel')):
    print(f"  {fn}")

In [2]:
import subprocess
result = subprocess.run(['find', '/', '-name', 'test.txt', '-path', '*/BestModel/*'],
                       capture_output=True, text=True, timeout=15)
print(result.stdout)

---
## 5. Summary

| Model | Dev F1 | Notes |
|-------|--------|-------|
| RoBERTa-base (baseline) | 0.48 | Standard CE loss, t=0.5 |
| **DeBERTa-v3-base (ours)** | >0.52 (target) | Keyword prepending + weighted sampling + sqrt pos_weight + multi-task + two-stage |

### Key design choices

1. **DeBERTa-v3-base over RoBERTa-base**: Disentangled attention encodes positional and content information separately, giving stronger representations for nuanced language like PCL. ELECTRA-style pretraining provides richer token-level supervision.

2. **max_length = 256**: PCL paragraphs are often long — truncating at 128 tokens cuts roughly half the text.

3. **Keyword prepending**: Each text is prefixed with `"Topic: {keyword}."` (e.g. `"Topic: homeless."`). The keyword column identifies which vulnerable group the paragraph discusses — a direct prior for PCL detection — and is available in all splits including test.

4. **WeightedRandomSampler (25% positive target)**: With a 9.5:1 imbalance, a random batch of 12 contains only ~1 PCL example. The sampler oversamples PCL so each batch has ~3 positive examples, giving the model 2.5× more signal per update without artificially replicating data.

5. **sqrt pos_weight ≈ 3.1**: Using `sqrt(neg/pos)` instead of the full ratio (9.55) avoids double-counting the imbalance already handled by the sampler. This produces a natural threshold close to 0.5, which generalises to the test set far better than the 0.840 threshold seen with pos_weight=9.55.

6. **Multi-task auxiliary loss** (weight=0.3): The 7 PCL sub-categories are predicted simultaneously. The auxiliary signal forces the encoder to learn fine-grained PCL features.

7. **Early stopping (patience=4)** + **two-stage training**: Stage 1 (train→dev) finds `best_epoch` and `best_thresh`. Stage 2 trains a fresh model on train+dev for `best_epoch` epochs, giving ~25% more labelled data for the final test predictions.

### Output files (`BestModel/`)

| File | Contents |
|------|----------|
| `dev.txt` | 2,094 binary predictions (0/1), one per line |
| `test.txt` | 3,832 binary predictions (0/1), one per line |
| `best_model.pt` | Stage-1 best checkpoint |
| `final_model.pt` | Stage-2 checkpoint (train + dev) — used for test predictions |